In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from data_collection import build_dataset
from data_selection import select_data, split_data, create_yolo_format, convert_labels_to_numeric_representation
import pandas as pd
from ultralytics import YOLO

/Users/josphslater/Programming/Personal/Pokemon-Visualization-Tool/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Function which pulls and organizes the images. Starts with pulling all the pokemon api, which takes the longest. Followed by kaggle then huggingface. On my machine it takes approximtely 3-4 minutes

In [3]:
build_dataset()

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 166560.81it/s]


Data collection completed


In [4]:
df, models = select_data(sources='all', levels=[1], exclude_3d=True, return_3d=True)

Selected 2018 images across 12 Pokemon.  (201 3D-model images excluded.)


In [6]:
train, test, val = split_data(df)

In [7]:
train.head()

,image_path,label,source
313,/Users/josphslater/Programming/Personal/Pokemo...,Snorlax,PokemonAPI
1138,/Users/josphslater/Programming/Personal/Pokemo...,Scyther,Kaggle
1025,/Users/josphslater/Programming/Personal/Pokemo...,Magikarp,Kaggle
440,/Users/josphslater/Programming/Personal/Pokemo...,Pikachu,PokemonAPI
1120,/Users/josphslater/Programming/Personal/Pokemo...,Snorlax,Kaggle


In [8]:
models.head()

,image_path,label,source
0,/Users/josphslater/Programming/Personal/Pokemo...,Eevee,PokemonAPI
1,/Users/josphslater/Programming/Personal/Pokemo...,Eevee,PokemonAPI
2,/Users/josphslater/Programming/Personal/Pokemo...,Eevee,PokemonAPI
3,/Users/josphslater/Programming/Personal/Pokemo...,Eevee,PokemonAPI
4,/Users/josphslater/Programming/Personal/Pokemo...,Eevee,PokemonAPI


This following code puts the data into a Ultralytics format for yolo models. It essentially tags each image with a bounding box (the whole image is the bounding box) needed for yolo models, and stores it in a text file. The images are also copied to keep referencing in an ultralytics yaml easier. 


In [ ]:
list_of_labels = None # us the follwing list to train for specific pokemon ["magikarp","pikachu", "meowth"]
print((df["label"].value_counts()))

numberic_labels_train = convert_labels_to_numeric_representation(train, list_of_labels)
numberic_labels_test = convert_labels_to_numeric_representation(test, list_of_labels)
numberic_labels_val = convert_labels_to_numeric_representation(val, list_of_labels)

create_yolo_format(train, 'train', numberic_labels_train)
create_yolo_format(test, 'test', numberic_labels_test)
create_yolo_format(val, 'val', numberic_labels_val)

In [ ]:
model = YOLO('yolov8n.pt')
model.train(data='../Dataset/My_yolo_dataset/pokedata.yaml', epochs=20, imgsz=640, device=[4,5], lr0=0.005)



In [ ]:
#This will save the predictions Pokemon-Visualization-Tool/data_operations/runs/detect/predictions/test_predictions
results = model.predict(source="../Dataset/My_yolo_dataset/test/images/", project="predictions/", name="test_predictions", conf=0.25, save=True , exist_ok=True)
